# ACC Framework V2.2-FJE — Comprehensive Results Analysis

This notebook restores the richer **legacy-style presentation layer** without reverting the current research design. The frozen scalar FJE objective and its four physical components remain the optimization evidence; TTC, observed TTE, time response, held-out motion, and physical robustness remain **validation-only** evidence.

The reusable implementation lives in `analysis_pipeline.py`. It writes deterministic tables/figures to `analysis_outputs/`, so paper figures are not trapped inside notebook state like tiny scientific hostages.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
OUT = ROOT / "analysis_outputs"
REGENERATE = True  # set False when you only want to inspect existing outputs

if REGENERATE:
    subprocess.run([sys.executable, "analysis_pipeline.py", "--clean"], check=True)
print("Analysis output:", OUT)

## 1. Objective-function values and physical decomposition

The active study has **one frozen scalar FJE cost**, so the legacy four-objective presentation is replaced by two complementary views: (1) scalar-cost stability/distribution across 30 runs and (2) the underlying cruise RMSE, following-spacing RMSE, realized jerk RMS, and positive tractive-energy intensity. Representative controllers are observed runs nearest each algorithm's median final fitness.

In [ ]:
display(pd.read_csv(OUT/"01_objective/objective_summary.csv"))
display(pd.read_csv(OUT/"01_objective/representative_controllers.csv"))
display(Image(filename=str(OUT/"01_objective/objective_distribution_boxplot.png")))
display(Image(filename=str(OUT/"01_objective/objective_stability_by_run.png")))
display(Image(filename=str(OUT/"01_objective/representative_cost_decomposition.png")))

## 2. Trajectory motion — periodic cruise, OpenACC, WLTC, sinusoidal following

Motion is shown using the observed-median controller from every optimizer. OpenACC includes both frozen calibration windows and held-out Casale; WLTC remains held-out/out-of-distribution validation. The synthetic periodic cruise and sinusoidal following cases are reporting diagnostics and never enter the frozen cost.

In [ ]:
for name in [
    "cruise_control_periodic.png",
    "c1_openacc_astazero_motion.png",
    "c2_openacc_vicolungo_motion.png",
    "v1_casale_motion.png",
    "v2_wltc_motion.png",
    "syn_follow_sinusoidal_motion.png",
]:
    display(Markdown(f"### {name.replace('_',' ').replace('.png','')}") )
    display(Image(filename=str(OUT/"04_motion"/name)))

## 3. TTC and observed TTE

TTC is computed from ground-truth evaluator states and remains outside controller inputs. The TTE table is explicitly an **observed/operational recovery diagnostic**, not the analytical TTE formula from the literature. Its exact disturbance and equilibrium thresholds are frozen in `08_audit/analysis_metadata.json`, avoiding the delightful research practice of changing thresholds until a figure behaves.

In [ ]:
display(pd.read_csv(OUT/"05_safety/ttc_and_following_summary.csv"))
display(pd.read_csv(OUT/"05_safety/tte_applicability.csv"))
display(pd.read_csv(OUT/"05_safety/observed_tte_summary.csv"))
display(Image(filename=str(OUT/"05_safety/ttc_v1_casale.png")))

## 4. Cruise-control time-response results

Every one of the ten set-speed transitions is evaluated independently for rise time, settling time, overshoot, peak time, and steady-state error. This is intentionally separate from the frozen FJE calibration objective.

In [ ]:
display(pd.read_csv(OUT/"06_time_response/cruise_time_response_summary.csv"))
for name in ["rise_time_by_transition.png","settling_time_by_transition.png","overshoot_pct_by_transition.png","steady_state_error_by_transition.png"]:
    display(Image(filename=str(OUT/"06_time_response"/name)))

## 5. Convergence

Convergence is summarized at every iteration using the median and interquartile range across independent runs, rather than displaying a conveniently photogenic single run.

In [ ]:
display(Image(filename=str(OUT/"02_convergence/convergence_median_iqr.png")))
display(pd.read_csv(OUT/"02_convergence/convergence_median_iqr.csv").head(18))

## 6. Statistical auditing pipeline

The audit runs on final scalar cost **and all four physical FJE components**. It provides Q–Q diagnostics, Shapiro–Wilk tests, independent-run Kruskal-Wallis and Mann-Whitney comparisons with Holm/Bonferroni corrections and Cliff’s delta. Run IDs alone are not statistical pairing.

In [ ]:
display(pd.read_csv(OUT/"03_statistics/shapiro_wilk.csv"))
display(pd.read_csv(OUT/"03_statistics/kruskal_wallis.csv"))
display(pd.read_csv(OUT/"03_statistics/independent_mann_whitney_corrected.csv"))

## 7. Physical-parameter robustness

The inherited 12-case stress matrix varies mass, rolling resistance, and ±10° road slope while keeping the representative controller fixed. These are **stress cases**, not claims about typical roads. Sensor robustness is deliberately not fabricated because the active manifest has no preregistered noise/bias perturbation grid.

In [ ]:
robust = pd.read_csv(OUT/"07_robustness/physical_robustness_metrics.csv")
display(robust)
for name in [
    "robustness_cruise_cruise_speed_rmse_mps.png",
    "robustness_following_spacing_rmse_m.png",
    "robustness_following_ttc_min_s.png",
    "robustness_following_positive_tractive_energy_wh_per_km.png",
    "robustness_casale_tte_median_s.png",
]:
    display(Image(filename=str(OUT/"07_robustness"/name)))

## 8. Reproducibility and consistency audit

The final gate checks run coverage, convergence-curve length, function-evaluation accounting, and exact reconstruction of the frozen scalar cost from the four weighted physical components.

In [ ]:
issues = pd.read_csv(OUT/"08_audit/consistency_issues.csv")
display(issues)
meta = __import__('json').loads((OUT/"08_audit/analysis_metadata.json").read_text())
display(pd.DataFrame([{
    "benchmark": meta["benchmark"],
    "framework_version": meta["framework_version"],
    "issue_count": meta["issue_count"],
    "max_cost_reconstruction_abs_error": meta["max_cost_reconstruction_abs_error"],
    "representative_policy": meta["representative_policy"],
}]))

## Interpretation guardrails

- Lower scalar FJE cost does not by itself identify *why* performance differs; use the physical-component tables and contribution figure.
- Statistical significance is not a substitute for effect magnitude; inspect the Cliff’s delta column and distribution plots.
- TTC/TTE, time-response metrics, WLTC/OpenACC held-out traces, and robustness tests are validation evidence and must not be retroactively folded into the already-frozen cost profile.
- `V3_LEGACY` remains available as a regression trajectory, but the primary manuscript-facing synthetic following plot is `SYN_FOLLOW_SINUSOIDAL` so its role is explicit.